<a href="https://colab.research.google.com/github/AbhayPSingh23/Group-DNA-Analyzer/blob/main/GroupDNA_Abhay_DS17729.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **GROUP DNA** : The WhatsApp Chat Analyzer

**Name**: Abhay Pratap Singh

**Roll Number**: DS17729

**Batch**: AUGUST-2026
# Date: 20-08-2026

# Feature 1:  THE PARSER


In [15]:
from datetime import datetime, timedelta
import numpy as np
import string

file_path = 'hostel_bois.txt'

with open(file_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

messages = []
sys_msgs = 0
media_msgs = 0
deleted_msgs = 0

for line in lines:
    line = line.strip()
    if not line:
        continue

    parts = line.split(' - ', 1)
    if len(parts) < 2:
        sys_msgs += 1
        continue

    timestamp_str, rest = parts
    try:
        # ASSUMING FORMAT DD/MM/YY, HH:MM
        dt = datetime.strptime(timestamp_str, '%d/%m/%y, %H:%M')
    except ValueError:
        try:
            # FALL BACK FOR 4 DIGIT YEAR (DD/MM/YYYY) JUST IN CASE
            dt = datetime.strptime(timestamp_str, '%d/%m/%Y, %H:%M')
        except ValueError:
            sys_msgs += 1
            continue

    sender_parts = rest.split(': ', 1)
    if len(sender_parts) < 2:
        sys_msgs += 1
        continue

    sender, msg_text = sender_parts

    is_media = False
    is_deleted = False

    if msg_text == '<Media omitted>':
        media_msgs += 1
        is_media = True
    elif msg_text == 'This message was deleted':
        deleted_msgs += 1
        is_deleted = True

    messages.append({
        'dt': dt,
        'sender': sender,
        'text': msg_text,
        'is_media': is_media,
        'is_deleted': is_deleted
    })

print(f"Successfully parsed {len(messages)} messages.")
print(f"Skipped {sys_msgs} system messages, counted {media_msgs} media omitted and {deleted_msgs} deleted messages.")
print("First 5 messages:")
for m in messages[:5]: print(m)



Successfully parsed 3174 messages.
Skipped 4 system messages, counted 32 media omitted and 15 deleted messages.
First 5 messages:
{'dt': datetime.datetime(2024, 4, 1, 1, 17), 'sender': 'Rahul', 'text': 'scene fix', 'is_media': False, 'is_deleted': False}
{'dt': datetime.datetime(2024, 4, 1, 1, 17), 'sender': 'Rahul', 'text': 'haan', 'is_media': False, 'is_deleted': False}
{'dt': datetime.datetime(2024, 4, 1, 1, 18), 'sender': 'Rahul', 'text': 'kya scene', 'is_media': False, 'is_deleted': False}
{'dt': datetime.datetime(2024, 4, 1, 2, 13), 'sender': 'Rahul', 'text': 'abhi free hai?', 'is_media': False, 'is_deleted': False}
{'dt': datetime.datetime(2024, 4, 1, 2, 13), 'sender': 'Rahul', 'text': 'abey', 'is_media': False, 'is_deleted': False}


## Feature 2: GROUP OVERVIEW


In [16]:
participants = set(m['sender'] for m in messages)
start_date = messages[0]['dt']
end_date = messages[-1]['dt']
total_days = (end_date - start_date).days + 1

msg_counts = {p: 0 for p in participants}
for m in messages:
    msg_counts[m['sender']] += 1

sorted_counts = sorted(msg_counts.items(), key=lambda x: x[1], reverse=True)
max_msgs = sorted_counts[0][1] if sorted_counts else 1

print("============================================================")
print("    GROUP OVERVIEW   ")
print("============================================================")
print(f" Group        : Hostel Bois 4ever")
print(f" Period       : {start_date.strftime('%d %B %Y')} to {end_date.strftime('%d %B %Y')} ({total_days} days)")
print(f" Total messages : {len(messages):,}")
print(f" Participants   : {len(participants)}")
print("\nMESSAGES PER PERSON")

for person, count in sorted_counts:
    pct = (count / len(messages)) * 100
    # 20 blocks max length for the bar chart
    bar_len = int((count / max_msgs) * 20)
    bar = '█' * bar_len

    # Adjust padding to align names, bars, and numbers perfectly
    print(f" {person:<14} {bar:<20} {count:>4} ({pct:>4.1f}%)")

    GROUP OVERVIEW   
 Group        : Hostel Bois 4ever
 Period       : 01 April 2024 to 30 May 2024 (60 days)
 Total messages : 3,174
 Participants   : 6

MESSAGES PER PERSON
 Rahul          ████████████████████  953 (30.0%)
 Priya          ███████████████       718 (22.6%)
 Neha           █████████████         635 (20.0%)
 Aman           ██████████            490 (15.4%)
 Karan          ███████               354 (11.2%)
 Vikas                                 24 ( 0.8%)


## Feature 3: MOST ACTIVE DAY AND HOUR


In [17]:
daily_counts = {}
hourly_counts = {}

for i in messages:
    d = i['dt'].date()
    h = i['dt'].hour
    daily_counts[d] = daily_counts.get(d, 0) + 1
    hourly_counts[h] = hourly_counts.get(h, 0) + 1

busiest_day = max(daily_counts, key=daily_counts.get)
busiest_hour = max(hourly_counts, key=hourly_counts.get)
avg_per_day_busiest_hour = hourly_counts[busiest_hour] / total_days

print(f"The Busiest day  : {busiest_day.strftime('%d %B %Y')} ({daily_counts[busiest_day]} messages)")
print(f"The Busiest hour : {busiest_hour:02d}:00 - {busiest_hour+1:02d}:00  (avg {int(avg_per_day_busiest_hour)} messages per day)")



The Busiest day  : 04 May 2024 (76 messages)
The Busiest hour : 18:00 - 19:00  (avg 4 messages per day)


## Feature 4: THE ACTIVITY HEATMAP (NumPy)


In [18]:
participant_list = [p for p, _ in sorted_counts]
heatmap = np.zeros((len(participant_list), 24), dtype=int)

for n in messages:
    p_idx = participant_list.index(n['sender'])
    h_idx = n['dt'].hour
    heatmap[p_idx, h_idx] += 1

print("ACTIVITY HEATMAP (hour of day, columns 00 to 23)")
print("               00 03 06 09 12 15 18 21")

for i, p in enumerate(participant_list):
    row = heatmap[i]
    max_val = max(row)
    if max_val == 0: max_val = 1

    blocks = []
    for val in row:
        pct = val / max_val
        # Removed trailing spaces from the blocks to compress the visual
        if pct == 0:
            blocks.append(' ')
        elif pct <= 0.25:
            blocks.append('.')
        elif pct <= 0.50:
            blocks.append('░')
        elif pct <= 0.75:
            blocks.append('▒')
        else:
            blocks.append('█')

    heatmap_str = " ".join(blocks)
    print(f" {p:<14} {heatmap_str}")

ACTIVITY HEATMAP (hour of day, columns 00 to 23)
               00 03 06 09 12 15 18 21
 Rahul          . . . . . . . . . . . . ▒ ░ ░ ▒ ▒ ░ █ ▒ ░ █ ▒ ▒
 Priya                      . ░ ▒ █ █ █ █ ▒ ▒ ░ ░ ▒ ▒ █ ▒ ░ ░ .
 Neha                     ░ . . ▒ █ █ ░ ▒ ▒ ░ . ▒ █ █ █ ▒ ░ ░ ░
 Aman           ▒ █ ▒ ▒ █                   . . . . . . . .   ▒
 Karan                        . ░ ░ ▒ ░ █ ▒ █ ▒ ▒ ▒ ▒ █ ▒ ░ . .
 Vikas                        ░ █ ░ ░   ▒ ▒   ░ ░ █ ▒ ▒ ░ ░ ░ ▒


## Feature 5: TOP WORDS


In [19]:
stop_words = {'i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for', 'it', 'my', 'me', 'you', 'that', 'this', 'hai', 'ki', 'se', 'ko', 'bhi', 'na', 'toh', 'ke', 'ka', 'ye', 'kya'}
word_counts = {}

for n in messages:
    if n['is_media'] or n['is_deleted']:
        continue

    text = n['text'].lower()
    for p in string.punctuation:
        text = text.replace(p, ' ')

    words = text.split()
    for w in words:
        if w not in stop_words and len(w) > 1:
            word_counts[w] = word_counts.get(w, 0) + 1

top_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:10]

print("THIS IS GROUP'S FAVOURITE WORDS")
for w, count in top_words:

    # 20 BLOCKS MAX LENGTH
    bar_len = int((count / top_words[0][1]) * 20)
    bar = '█' * bar_len
    print(f" {w:<10} {bar:<20} {count}")



THIS IS GROUP'S FAVOURITE WORDS
 was        ████████████████████ 385
 how        ████████████████     321
 guys       ████████████████     318
 today      ███████████████      292
 so         ███████████████      292
 about      ██████████████       274
 am         █████████████        260
 at         █████████████        257
 he         ███████████          220
 his        ███████████          217


## Feature 6: RESPONSE SPEED AND THE SILENT STREAKS


In [20]:
response_times = {p: [] for p in participant_list}
silent_streaks = {p: 0 for p in participant_list}

last_sender = None
last_dt = None

# FOR RESPONSE TIME

for j in messages:
    if last_sender and last_sender != j['sender']:
        gap = (j['dt'] - last_dt).total_seconds()
        response_times[j['sender']].append(gap)
    last_sender = j['sender']
    last_dt = j['dt']

avg_responses = {}
for p, gaps in response_times.items():
    if gaps:
        avg_responses[p] = sum(gaps) / len(gaps)
    else:
        avg_responses[p] = 0

# FOR SILENT STREAKS

date_set = {m['dt'].date() for m in messages}
all_dates = sorted(list(date_set))

for p in participant_list:
    max_streak = 0
    current_streak = 0
    p_dates = {m['dt'].date() for m in messages if m['sender'] == p}

    for d in all_dates:
        if d not in p_dates:
            current_streak += 1
            if current_streak > max_streak:
                max_streak = current_streak
        else:
            current_streak = 0
    silent_streaks[p] = max_streak

fastest = min([p for p in avg_responses.items() if p[1] > 0], key=lambda x: x[1])
slowest = max([p for p in avg_responses.items() if p[1] > 0], key=lambda x: x[1])

print(" THE RESPONSE PATTERNS")
print(f"Fastest replier : {fastest[0]} (avg {fastest[1]/60:.1f} minutes)")
print(f"Slowest replier : {slowest[0]} (avg {slowest[1]/3600:.1f} hours)")

print("LONGEST SILENT STREAKS")
for p, streak in sorted(silent_streaks.items(), key=lambda x: x[1], reverse=True):
    print(f" {p:<10} : {streak} days")



 THE RESPONSE PATTERNS
Fastest replier : Rahul (avg 34.9 minutes)
Slowest replier : Aman (avg 0.9 hours)
LONGEST SILENT STREAKS
 Vikas      : 11 days
 Rahul      : 0 days
 Priya      : 0 days
 Neha       : 0 days
 Aman       : 0 days
 Karan      : 0 days


## Feature 7: PERSONALITY ARCHETYPE DETECTION


In [21]:
archetypes = {p: [] for p in participant_list}

# 1. THE SPAMMER : Avg consecutive message burst > 3
bursts = {p: [] for p in participant_list}
current_burst = 0
curr_sender = None

for m in messages:
    if m['sender'] == curr_sender:
        current_burst += 1
    else:
        if curr_sender:
            bursts[curr_sender].append(current_burst)
        curr_sender = m['sender']
        current_burst = 1
if curr_sender: bursts[curr_sender].append(current_burst)

spammer_scores = {p: (sum(b)/len(b) if b else 0) for p, b in bursts.items()}

# 2. THE QUESTION MASTER : end with ?
qm_scores = {p: 0 for p in participant_list}
for k in messages:
    if not k['is_media'] and not k['is_deleted']:
        if k['text'].strip().endswith('?'):
            qm_scores[k['sender']] += 1
qm_pct = {p: (qm_scores[p] / msg_counts[p] * 100) for p in participant_list}


# 3. THE NIGHT OWL : > 60% messages between 23-04
owl_scores = {p: 0 for p in participant_list}
for i, p in enumerate(participant_list):
    night_msgs = sum(heatmap[i, 23:]) + sum(heatmap[i, 0:5])
    total_msgs = sum(heatmap[i, :])
    owl_scores[p] = (night_msgs / total_msgs * 100) if total_msgs > 0 else 0

# 4. THE STORYTELLER : Avg words per message > 30
words_per_msg = {p: [] for p in participant_list}
for m in messages:
    if not m['is_media'] and not m['is_deleted']:
        words_per_msg[m['sender']].append(len(m['text'].split()))
storyteller_scores = {p: (sum(w)/len(w) if w else 0) for p, w in words_per_msg.items()}

# 5. THE DRAMA QUEEN : > 30% messages all-caps or 2+ !
drama_scores = {p: 0 for p in participant_list}
for j in messages:
    if not j['is_media'] and not j['is_deleted']:
        t = j['text']
        if (t.isupper() and len(t) > 3) or t.count('!') >= 2:
            drama_scores[j['sender']] += 1
drama_pct = {p: (drama_scores[p] / msg_counts[p] * 100) for p in participant_list}

# 6. THE GHOST : Silent > 60% of days
ghost_scores = {p: 0 for p in participant_list}
for p in participant_list:
    days_active = len({m['dt'].date() for m in messages if m['sender'] == p})
    days_silent = total_days - days_active
    ghost_scores[p] = (days_silent / total_days * 100)

# 7. THE COMEDIAN : 'lol', 'lmao', etc.
haha_words = ['lol', 'lmao', 'haha', 'rofl', 'lmfao']
comedian_scores = {p: 0 for p in participant_list}
for m in messages:
    if not m['is_media'] and not m['is_deleted']:
        t = m['text'].lower()
        if any(hw in t for hw in haha_words):
            comedian_scores[m['sender']] += 1
comedian_pct = {p: (comedian_scores[p] / msg_counts[p] * 100) for p in participant_list}

# 8. THE GROUP MOM : caring keywords
caring_words = ['okay', 'safe', 'eat', 'sleep', 'take care', 'are you', 'please', 'reminder', 'drink water', "don't forget"]
mom_scores = {p: 0 for p in participant_list}
for n in messages:
    if not n['is_media'] and not n['is_deleted']:
        t = n['text'].lower()
        if any(cw in t for cw in caring_words):
            mom_scores[n['sender']] += 1

# [Keep all the scoring logic (1 through 8) exactly the same as your original script above this point]

# ASSIGN ARCHETYPES (Priority-based evaluation)
final_archetypes = {}
for p in participant_list:

   # Priority cascade: The first threshold the user passes defines their archetype
    if ghost_scores[p] > 60:
        best_arch = 'THE GHOST'
        reason = f"silent on {int((ghost_scores[p]/100)*total_days)} of {total_days} days"

    elif spammer_scores[p] > 3:
        best_arch = 'THE SPAMMER'
        reason = f"avg {spammer_scores[p]:.1f} msgs in a row"

    elif drama_pct[p] > 30:
        best_arch = 'THE DRAMA QUEEN'
        reason = f"{drama_pct[p]:.1f}% ALL-CAPS messages"

    elif storyteller_scores[p] > 30:
        best_arch = 'THE STORYTELLER'
        reason = f"avg {storyteller_scores[p]:.1f} words per msg"

    elif owl_scores[p] > 60:
        best_arch = 'THE NIGHT OWL'
        reason = f"{owl_scores[p]:.1f}% msgs between 23h-04h"

    # Explicitly check for Group Mom before Question Master
    elif mom_scores[p] > 50:
        best_arch = 'THE GROUP MOM'
        reason = f"caring keyword score: {mom_scores[p]}"

    elif qm_pct[p] > 25:
        best_arch = 'THE QUESTION MASTER'
        reason = f"{qm_pct[p]:.1f}% messages with ?"

    elif comedian_pct[p] > 10:
        best_arch = 'THE COMEDIAN'
        reason = f"{comedian_pct[p]:.1f}% haha/lol messages"

    else:
        best_arch = 'THE NORMIE'
        reason = "balanced participation"

    final_archetypes[p] = (best_arch, reason)

print("PERSONALITY ARCHETYPES")
for p, (arch, reason) in final_archetypes.items():
    print(f" {p:<14} -> {arch:<18} ({reason})")

print("============================================================")
print("Generated by GroupDNA • Built with Python + NumPy")
print("============================================================")

PERSONALITY ARCHETYPES
 Rahul          -> THE SPAMMER        (avg 4.5 msgs in a row)
 Priya          -> THE GROUP MOM      (caring keyword score: 459)
 Neha           -> THE DRAMA QUEEN    (62.2% ALL-CAPS messages)
 Aman           -> THE NIGHT OWL      (79.8% msgs between 23h-04h)
 Karan          -> THE STORYTELLER    (avg 57.0 words per msg)
 Vikas          -> THE GHOST          (silent on 44 of 60 days)
Generated by GroupDNA • Built with Python + NumPy


The report `whatsapp_chat_analysis_report.png` has been generated and saved in your current Colab environment. You can find it in the files browser (folder icon on the left panel).